In [6]:
# Load necessary libraries
suppressMessages(library(DESeq2))
suppressMessages(library(ggplot2))
suppressMessages(library(latex2exp))
suppressMessages(library(pROC))
suppressMessages(library(data.table))
suppressMessages(library(glmnet))
suppressMessages(library(corrplot))

In [22]:
exp_ <- read.csv("data/GSE152004_695_expr_norm.txt", sep="\t")
raw <- read.csv("data/GSE152004_695_raw_counts.txt", sep="\t")

In [23]:
write.csv(exp_, "data/GSE152004_normalized.csv")
write.csv(raw, "data/GSE152004_raw.csv")

In [12]:
meta <- read.csv("data/meta.csv")
write.csv(meta, "data/condition_GSE152004.csv", row.names = FALSE)

In [20]:
head(cts_normalized)

numeric(0)

In [15]:
# Clear the workspace
rm(list = ls())

# Load raw count and normalized count data
cts <- as.matrix(read.csv("data/GSE152004_raw.csv"))
rownames(cts) <- cts[,1]
cts <- cts[,-1]
mode(cts) <- "numeric"

cts_normalized <- as.matrix(read.csv("data/GSE152004_normalized.csv"))
rownames(cts_normalized) <- cts_normalized[,1]
cts_normalized <- cts_normalized[,-1]
mode(cts_normalized) <- "numeric"

# Set seed for reproducibility
set.seed(461)

# Split data into training and testing sets
sample_test <- sample(colnames(cts), 0.2 * ncol(cts))
sample_train <- setdiff(colnames(cts), sample_test)
cts_test <- cts[, sample_test]
cts_train <- cts[, sample_train]
cts_test_normalized <- cts_normalized[, sample_test]
cts_train_normalized <- cts_normalized[, sample_train]

# Load sample condition metadata
coldata <- read.csv("data/condition_GSE152004.csv", row.names=1)
Y_train <- coldata[sample_train,]
Y_test <- coldata[sample_test,]
coldata$Condition <- factor(coldata$Condition)
coldata_train <- coldata[sample_train,,drop=FALSE]
coldata_test <- coldata[sample_test,,drop=FALSE]
colnames(coldata_train) <- colnames(coldata_test) <- "condition"

# Ensure data consistency
stopifnot(all(rownames(coldata_train) %in% colnames(cts_train)))
stopifnot(all(rownames(coldata_train) == colnames(cts_train)))

# Create DESeq2 dataset and perform differential expression analysis
dds <- DESeqDataSetFromMatrix(countData = cts_train,
                              colData = coldata_train,
                              design = ~ condition)

dds <- dds[rowSums(counts(dds)) >= 10,]
dds <- DESeq(dds)
res <- results(dds)
res <- res[order(res$log2FoldChange),]

# Save results
write.csv(res, "DESeq_results.csv")

# Generate Volcano plot
jpeg("volcano_plot.jpg", units="in", width=5, height=5, res=300)
with(res, plot(log2FoldChange, -log10(pvalue), pch=20, 
               main="Volcano Plot", xlim=c(-1.5,1.5), ylim=c(0,9), cex=0.1,
               ylab=TeX("-log_{10}(adjusted p-value)"), 
               xlab=TeX("log_{2}(Fold Change)")))
with(subset(res, padj < 0.05), points(log2FoldChange, -log10(pvalue), 
                                      pch=20, col="blue", cex=0.38))
dev.off()

# Generate MA plot
jpeg("MA_plot.jpg", units="in", width=5, height=5, res=300)
plotMA(res, ylim=c(-1.5,1.5), alpha=0.05, main="MA Plot", cex=0.38,
       ylab=TeX("log_{2}(Fold Change)"))
dev.off()

# LASSO logistic regression for feature selection
lam.vec <- seq(0.01, 0.05, 0.001)
n.lam <- length(lam.vec)
auc <- numeric(n.lam)
mcc <- numeric(n.lam)
acc <- numeric(n.lam)

for(j in seq_along(lam.vec)) {
  fit_lasso <- glmnet(log(t(cts_train_normalized)[, rownames(res)] + 1), 
                      Y_train, family="binomial", lambda=lam.vec[j])
  Y_hat <- predict(fit_lasso, log(t(cts_test_normalized)[, rownames(res)] + 1), 
                   type="response", lambda=0.04)
  eval <- Evaluation(Y_test, 1 * (Y_hat > 0.5))
  m <- pROC::roc(Y_test, Y_hat, quiet = TRUE)
  auc[j] <- as.numeric(m$auc)
  mcc[j] <- eval$MCC
  acc[j] <- eval$ACC
}

# Select best lambda based on accuracy
best_lambda <- lam.vec[which.max(acc)]
fit_lasso <- glmnet(log(t(cts_train_normalized)[, rownames(res)] + 1), 
                    Y_train, family="binomial", lambda=best_lambda)

# Save predicted values
Y_hat_FC <- predict(fit_lasso, log(t(cts_test_normalized)[, rownames(res)] + 1),
                    s="lambda.min")
save(Y_hat_FC, file="Y_hat_FC.RData")

# Generate correlation heatmap
cor_data <- cor(log(t(cts_train_normalized[rownames(res), ]) + 1))
jpeg("correlation_heatmap.jpg", units="in", width=7, height=7, res=300)
corrplot(cor_data, method="color", tl.col="black", tl.srt=63, tl.cex=0.52)
dev.off()


ERROR: Error in cts_normalized[, sample_test]: incorrect number of dimensions
